In [8]:
import functools
import os
import sys
import json

from ultralytics import YOLO
import pandas as pd

sys.path.append("..")
import ultralytics_validator_wrapper
from tflite_inference import TFLiteInferer
from tflite_inference_input_adapter import TFLiteInferenceWithInputAdapter

pd.options.display.float_format = '{:.3f}'.format

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
BASE_EXPERIMENT_DIR = "../results/cam3_revB_batch1_7_margin_15_yolov8n_bg_wb_alif_clahe_box_01_dfl_01_2nd"
BASE_WEIGHTS_DIR = os.path.join(BASE_EXPERIMENT_DIR, "weights")
WEIGHTS_PTH = os.path.join(BASE_WEIGHTS_DIR, "best.pt")

HOLDOUT_PTH = " /home/klymy/data/cam4_revB/yolo_dataset/batch1_7_margin_15_bg_wb_alif_clahe/project_holdout.yaml"
VAL_PTH = " /home/klymy/data/cam4_revB/yolo_dataset/batch1_7_margin_15_bg_wb_alif_clahe/project.yaml"
POC_PTH = "/home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/project_pg.yaml"

if os.path.exists(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json")):
    with open(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json"), "r") as f:
        data = json.load(f)
    CONF_TH = float(data["conf_th"])
    IOU_TH = float(data["iou_th"])
else:
    print("No inference_params.json found, using default thresholds.")
    CONF_TH = 0.35
    IOU_TH = 0.50

args = dict(
    task="pose",
    device=[0],
    batch=128,
    verbose=False,
    conf=CONF_TH,
    iou=IOU_TH,
    imgsz=320,
    model="val",
    rect=False,
    single_cls=False,
)

In [10]:
model = YOLO(WEIGHTS_PTH)

In [11]:
inferer = TFLiteInferer(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
    head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

# inferer_adapter = TFLiteInferenceWithInputAdapter(
#     yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
#     head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
#     adapter_pth="../input_adapter/input_adapter_full_integer_quant.tflite",
#     conf_th=CONF_TH,
#     iou_th=IOU_TH,
#     imgsz=320,
# )

In [12]:
args["data"] = VAL_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_7_margin_15_bg_wb_alif_clahe/labels/val.cache... 470 images, 1 backgrounds, 0 corrupt: 100%|██████████| 470/470 [00:00<?, ?it/s]


tst validation: 100%|██████████| 4/4 [00:19<00:00,  4.93s/it]


                   all        470       2347      0.984      0.982       0.99      0.845      0.985      0.981      0.991       0.99
   task    F1     P     R  MAP@50
0   box 0.983 0.984 0.982   0.990
1  pose 0.983 0.985 0.981   0.991


In [13]:
args["data"] = HOLDOUT_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_7_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:06<00:00,  3.17s/it]


                   all        148        409       0.93      0.883      0.932      0.728      0.946      0.897      0.946      0.942
   task    F1     P     R  MAP@50
0   box 0.906 0.930 0.883   0.932
1  pose 0.921 0.946 0.897   0.946


In [14]:
args["data"] = POC_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/labels/val.cache... 191 images, 5 backgrounds, 0 corrupt: 100%|██████████| 191/191 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:08<00:00,  4.05s/it]


                   all        191        417      0.768      0.712      0.777      0.578      0.788      0.731      0.805      0.796
   task    F1     P     R  MAP@50
0   box 0.739 0.768 0.712   0.777
1  pose 0.759 0.788 0.731   0.805
